# Analyse et plan de correction — catégories de défauts non détectées (`pill`)

Le notebook 06 a montré qu'au seuil retenu (percentile 99), 4 catégories de défauts
sur 7 ont un rappel de 0 % : `contamination`, `crack`, `faulty_imprint`, `scratch`.
Ce notebook restitue le diagnostic mené pour comprendre pourquoi, et présente le plan
de correction en deux phases, implémentées dans les notebooks 08 (Phase 1, sans
réentraînement) et 09 (Phase 2, PatchCore).

**Aucun code de production ici** : ce notebook ne fait que diagnostiquer (avec du
code réel, recalculé à partir du modèle déjà entraîné — pas de chiffres recopiés en
dur) et présenter le plan.

In [1]:
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display

from indusense.vision.anomaly import (
    calibrate_threshold,
    error_maps,
    healthy_baseline,
    image_auroc,
    image_scores,
    image_scores_pooled,
    load_masks,
    reconstruct,
)
from indusense.vision.dataset import (
    DEFAULT_IMAGE_SIZE,
    list_defects,
    load_defect_images,
    load_good_images,
    train_val_split,
)
from indusense.vision.train import load_trained_model

IMAGE_SIZE = DEFAULT_IMAGE_SIZE
FIGURES_DIR = Path("../reports/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

model = load_trained_model()
defects = list_defects()

good_train = load_good_images("train", IMAGE_SIZE)
_train_raw, val_good = train_val_split(good_train, val_fraction=0.15, seed=42)
maps_val = error_maps(val_good, reconstruct(model, val_good))

display(Markdown(f"- **Modèle chargé** : {model.count_params()} paramètres."))

- **Modèle chargé** : 57235 paramètres.

# Section 1 — Diagnostic

## Ce n'est pas un problème de taille de défaut

À résolution native (800×800, avant redimensionnement à 256×256), les 4 catégories en
échec ont des zones de défaut comparables ou plus grandes que `color` (la catégorie
la mieux détectée hors `pill_type`, cas trivial). L'hypothèse « défauts trop fins,
perdus au redimensionnement » est donc écartée par les données — vérifié en mesurant
la boîte englobante du masque de vérité terrain à résolution native pour un exemple
de chaque catégorie.

## C'est un problème de contraste/nature du signal

Mesure directe de l'erreur de reconstruction à l'intérieur vs à l'extérieur du masque
de vérité terrain, pour chaque catégorie de défaut :

In [2]:
defect_images = {defect: load_defect_images(defect, IMAGE_SIZE) for defect in defects}
ratios = {}
for defect, images in defect_images.items():
    masks = load_masks(defect, IMAGE_SIZE)
    maps = error_maps(images, reconstruct(model, images))
    flat_maps = maps.reshape(len(maps), -1)
    mask_flat = masks.reshape(len(masks), -1)
    inside = np.array(
        [flat_maps[i][mask_flat[i]].mean() if mask_flat[i].sum() > 0 else np.nan for i in range(len(images))]
    )
    outside = np.array([flat_maps[i][~mask_flat[i]].mean() for i in range(len(images))])
    ratios[defect] = float(np.nanmean(inside) / np.nanmean(outside))

lines = [f"- **{defect}** : ratio ×{ratio:.1f}" for defect, ratio in sorted(ratios.items(), key=lambda x: -x[1])]
display(Markdown(chr(10).join(lines)))

- **color** : ratio ×67.8
- **combined** : ratio ×10.9
- **crack** : ratio ×6.7
- **pill_type** : ratio ×5.6
- **contamination** : ratio ×2.9
- **scratch** : ratio ×2.5
- **faulty_imprint** : ratio ×2.0

`color` est un écart de teinte/intensité franc (petites taches rouges) que la perte
MSE capture facilement même sur une petite zone. Les 4 catégories en échec sont des
altérations de **texture/structure à faible contraste** (fissure fine, tache proche du
ton de la pièce, gravure altérée, rayure fine) : leur élévation d'erreur, même mesurée
strictement à l'intérieur du défaut, reste proche du niveau de bruit de reconstruction
normal du modèle — pas seulement un problème d'agrégation du score, un signal
intrinsèquement faible à la source.

**Complication supplémentaire** : les pièces saines elles-mêmes ont des zones d'erreur
systématiquement élevées (contour ovale, gravure embossée « FF », imparfaitement
reconstruits par le bottleneck ×12 sur *toute* pièce saine, cf. notebook 03). Un score
basé sur le maximum brut ne peut pas distinguer un pic de défaut faible de ces zones
saines à erreur naturellement élevée.

## Six approches testées

Six façons de calculer le score d'anomalie ont été comparées, en gardant le même
modèle (aucun réentraînement). Les deux premières sont recalculées ci-dessous en
direct ; les quatre suivantes ont été mesurées lors du diagnostic (résultats
rapportés tels quels, non recalculés ici par souci de concision — certaines sont des
échecs délibérément non retenus dans le code du projet).

In [3]:
test_good = load_good_images("test", IMAGE_SIZE)
test_images = np.concatenate([test_good, *defect_images.values()], axis=0)
labels = np.concatenate(
    [np.zeros(len(test_good))] + [np.ones(len(images)) for images in defect_images.values()]
)
maps_test = error_maps(test_images, reconstruct(model, test_images))

# Approche 1 : score actuel (moyenne globale)
scores_val_1 = image_scores(maps_val)
threshold_1 = calibrate_threshold(scores_val_1, method="percentile", q=99.0)
scores_test_1 = image_scores(maps_test)

# Approche 5 : score retenu en Phase 1 (centile 99,5 après soustraction de la
# référence saine)
baseline = healthy_baseline(maps_val)
scores_val_5 = image_scores_pooled(maps_val, baseline, q=99.5)
threshold_5 = calibrate_threshold(scores_val_5, method="percentile", q=99.0)
scores_test_5 = image_scores_pooled(maps_test, baseline, q=99.5)

lines = ["- **catégorie** : rappel approche 1 (actuelle) vs approche 5 (Phase 1)"]
offset = len(test_good)
for defect in defects:
    n = len(defect_images[defect])
    r1 = (scores_test_1[offset : offset + n] > threshold_1).mean()
    r5 = (scores_test_5[offset : offset + n] > threshold_5).mean()
    lines.append(f"- **{defect}** : {r1:.1%} vs {r5:.1%}")
    offset += n
lines.append(
    f"- **AUROC image-level** : {image_auroc(scores_test_1, labels):.3f} vs "
    f"{image_auroc(scores_test_5, labels):.3f}"
)
display(Markdown(chr(10).join(lines)))

- **catégorie** : rappel approche 1 (actuelle) vs approche 5 (Phase 1)
- **color** : 40.0% vs 72.0%
- **combined** : 29.4% vs 70.6%
- **contamination** : 0.0% vs 0.0%
- **crack** : 0.0% vs 3.8%
- **faulty_imprint** : 0.0% vs 5.3%
- **pill_type** : 100.0% vs 100.0%
- **scratch** : 0.0% vs 0.0%
- **AUROC image-level** : 0.688 vs 0.756

**Quatre autres approches mesurées lors du diagnostic** (non recalculées ci-dessus) :

- **Approche 2 — maximum brut de la carte d'erreur** : 0 à 4,8 % de rappel sur les 4
  catégories cibles — inefficace, noyé dans le bruit de fond sain (contour/gravure,
  cf. complication ci-dessus).
- **Approche 3 — percentile élevé (99,9) sans référence saine** : 27 à 43 % en
  apparence, mais mesure non fiable — comparée à un seuil scalaire unique, sans
  contrôle du taux de fausses alertes (contrairement aux approches 1/5 ci-dessus, où
  le seuil est calibré et le taux de fausses alertes vérifié).
- **Approche 4 — score normalisé par (moyenne ET écart-type) par pixel de la
  validation saine** : **échec catastrophique**, 100 % de fausses alertes sur les
  saines de test. Avec seulement ~40 images de validation, l'écart-type par pixel
  (65 536 positions indépendantes) est bien trop bruité pour généraliser — un piège
  classique de sur-ajustement en haute dimension avec peu d'exemples. **Écartée**, au
  profit de l'approche 5 (soustraction de moyenne seule, sans écart-type).
- **Approche 6 — modèle entraîné en perte SSIM** (rechargeable depuis MLflow sans
  réentraînement) **et carte de dissimilarité SSIM locale par fenêtre** : dans les
  deux cas, aucune amélioration mesurable sur les 4 catégories cibles — les ratios
  intérieur/extérieur du masque restent du même ordre ou légèrement inférieurs à la
  MSE, et le bruit de fond sain est même plus élevé pour le modèle SSIM.
  L'hypothèse « SSIM détecterait mieux la texture » ne se vérifie pas avec ce modèle.

**Conclusion factuelle** : le modèle actuel (256×256, MSE, bottleneck ×12) produit,
pour les 4 catégories cibles, un signal de reconstruction intrinsèquement trop proche
de son propre bruit de fond — quelle que soit la façon dont on le mesure ou l'agrège
a posteriori. Une correction du score seul (Phase 1, approche 5) ne peut qu'améliorer
les catégories déjà partiellement détectées (`color`, `combined` ci-dessus) ; résoudre
les 4 catégories cibles demande une approche différente (Phase 2).

# Section 2 — Plan de correction

## Phase 1 (notebook 08) — sans réentraînement

Implémente l'approche 5 ci-dessus (`healthy_baseline` + `image_scores_pooled`,
ajoutées à `anomaly.py`) : un score basé sur un centile élevé de l'erreur, après
soustraction d'une référence saine. **Gain réel** sur `color`/`combined` et l'AUROC
global, **sans nouvelle fausse alerte** — mais ne corrige pas les 4 catégories
cibles, conformément au diagnostic ci-dessus (un problème de signal, pas seulement
d'agrégation du score).

## Phase 2 (notebook 09) — changement de paradigme : PatchCore

Puisque le signal de reconstruction lui-même est trop faible pour ces 4 catégories,
la correction ne peut pas venir d'un ajustement du score — il faut une approche qui
ne dépende pas de la reconstruction de l'auto-encodeur.

**Alternatives considérées** (discutées avec l'utilisateur avant de choisir) :
- **PaDiM** : features pré-entraînées + distribution gaussienne par position de
  patch (moyenne + covariance), distance de Mahalanobis en test.
- **PatchCore** *(retenu)* : features pré-entraînées + banque de mémoire des patches
  sains + distance au plus proche voisin. Considéré comme état de l'art sur MVTec AD,
  particulièrement fort sur les défauts de texture fine — exactement le type de
  défaut qui pose problème ici.
- **STFPM** (distillation enseignant-élève), **Deep SVDD** (classification
  one-class), **CutPaste** (auto-supervision par défauts synthétiques), **GANomaly**
  (GAN) : autres familles écartées au profit de PatchCore, jugé le meilleur compromis
  simplicité d'implémentation / performance démontrée sur ce type de défaut.

**Pourquoi ces méthodes peuvent aider là où l'auto-encodeur échoue** : elles
s'appuient sur un réseau **pré-entraîné sur des millions d'images** (ImageNet), dont
les features encodent déjà une notion riche de texture/structure — alors que
l'auto-encodeur de ce TP a été entraîné *from scratch* sur seulement 227 images
saines, probablement insuffisant pour apprendre une représentation aussi fine de la
texture.

**Résultats réels** : voir notebook 09 pour l'implémentation et l'évaluation
chiffrée de PatchCore, avec comparaison directe à la Phase 1 sur les 4 catégories
cibles.